# Mapas de ressonância: leitura exclusiva de `visco_final` no Drive, resultados em `visco_final/resultados`.
# Código completo em uma célula; modelo elástico incluído, sem arquivos `.py` externos.


In [ ]:
# -*- coding: utf-8 -*-
"""
MAPAS DE RESSONANCIA - ELASTICO, KELVIN-VOIGT E MAXWELL

SCRIPT AUTOCONTIDO: todas as rotinas do modelo elastico usadas para calcular
N_res estao abaixo, copiadas do elastico(1).py enviado pela autora.
NAO precisa encontrar, importar, baixar ou executar 'elastico.py'.

A entrada dos modelos Voigt e Maxwell continua sendo o checkpoint NPZ:
nao e possivel recriar esses dados sem os modelos de perdas e seus parametros.
Os dados sao lidos SOMENTE de PASTA (visco_final) e suas subpastas;
as figuras e tabelas vao para visco_final/resultados. Se faltar algum
checkpoint, avisa sem buscar caminhos antigos e sem inventar dados.

Elastico: mapa eta=1.16 + fronteiras de eta=2, 5, 10 (eta exato).
Voigt e Maxwell: tres mapas separados por faixas de eta, sem fronteiras.
As figuras sao salvas em PNG, PDF e EPS, portugues e ingles.
"""

from pathlib import Path
import os
import signal
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.special import spherical_jn, spherical_yn
from scipy.optimize import newton
from scipy.signal import find_peaks, peak_widths
from scipy.spatial import cKDTree
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# ---------------- pastas: dados e resultados ----------------
# Google Drive: Meu Drive > visco_final (uma subpasta por modelo, saída do gerador)
try:
    from google.colab import drive
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB:
    drive.mount("/content/drive")
    PASTA = "/content/drive/MyDrive/visco_final"
    SAIDA = os.path.join(PASTA, "resultados")
else:
    PASTA = os.path.abspath(os.environ.get(
        "VISCO_DADOS", os.path.join(os.path.expanduser("~"), "Downloads", "visco_final")))
    SAIDA = os.path.abspath(os.environ.get("VISCO_SAIDA", "resultados"))

FIG_DIR = os.path.join(SAIDA, "figuras")
TAB_DIR = os.path.join(SAIDA, "tabelas")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

LARGURA_BIN = 0.25
LIMITES_MAPA = {
    'elastico': ((-3.0, 3.5), (-3.0, 3.0)),
    'voigt': ((-3.0, 3.5), (-3.0, 3.0)),
    'maxwell': ((-3.0, 3.5), (-4.0, 3.0)),
}
ETA_FAIXAS = [(2/np.sqrt(3), 2.0), (2.0, 5.0), (5.0, np.inf)]
ROT_FAIXA = [
    r'$\eta\in[2/\sqrt{3};\,2)$',
    r'$\eta\in[2;\,5)$',
    r'$\eta\in[5;\,\infty)$',
]
ETA_EXATO_TOL = 1e-5
COBERTURA_MINIMA = 0.80
L_MAX = 10
TEMPO_LIMITE_CURVA = 60.0
SALVAR_CADA = 10
GERAR_INGLES = True
MOSTRAR_FIGURAS = True
PREENCHER_VAZIOS_BANCO = True  # reproduz o preenchimento do mapa enviado
CELL_LINECOLOR, CELL_LINEWIDTH = 'white', 0.42
BOX_COLOR = '#600B00'
FRONTEIRAS_ESTILOS = {
    2.0: dict(color='#8b3f2b', linestyle=(0,(5,3)), linewidth=1.5),
    5.0: dict(color='#3b82c4', linestyle=(0,(3,2)), linewidth=1.5),
    10.0: dict(color='#444444', linestyle=(0,(6,2,1.2,2)), linewidth=1.7),
}
ESTILO_FRONTEIRA_BASE = dict(color='black', linestyle='-', linewidth=1.2)
GUIAS = dict(color='0.85', linestyle=':', linewidth=1.0, zorder=3)
RESONANCE_CMAP = LinearSegmentedColormap.from_list(
    'ressonancias', ['white','#D9E1EA','#526F92','black'], N=256
)
RESONANCE_CMAP.set_bad('#D9D9D9')

# =========================================================
# MODELO FISICO ELASTICO ORIGINAL (FUNCOES USADAS PELO DETECTOR)
# =========================================================


CM_PADRAO = 1500.0


RHOM_PADRAO = 1000.0


class CurveTimeoutError(TimeoutError):
    pass


def _timeout_handler(signum, frame):
    raise CurveTimeoutError("Tempo maximo da curva excedido.")


class curve_timeout:
    """Context manager Linux/Unix para limite de wall-clock por curva."""
    def __init__(self, seconds):
        self.seconds = float(seconds)
        self.old_handler = None

    def __enter__(self):
        if self.seconds > 0.0:
            if not hasattr(signal, "SIGALRM"):
                raise RuntimeError("Timeout por SIGALRM requer Linux/Unix.")
            self.old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
            signal.setitimer(signal.ITIMER_REAL, self.seconds)
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.seconds > 0.0:
            signal.setitimer(signal.ITIMER_REAL, 0.0)
            if self.old_handler is not None:
                signal.signal(signal.SIGALRM, self.old_handler)
        return False


def J(n, x):
    if n >= 0:
        return spherical_jn(n, x)
    m = -n - 1
    return (-1)**(m + 1) * spherical_yn(m, x)


def dJ(n, x):
    if n >= 0:
        return spherical_jn(n, x, derivative=True)
    m = -n - 1
    return (-1)**(m + 1) * spherical_yn(m, x, derivative=True)


def H(n, x):
    if n >= 0:
        return spherical_jn(n, x) + 1j * spherical_yn(n, x)
    m = -n - 1
    return 1j * (-1)**m * (spherical_jn(m, x) + 1j * spherical_yn(m, x))


def dH(n, x):
    if n >= 0:
        return spherical_jn(n, x, derivative=True) + 1j * spherical_yn(n, x, derivative=True)
    m = -n - 1
    return 1j * (-1)**m * (
        spherical_jn(m, x, derivative=True) + 1j * spherical_yn(m, x, derivative=True)
    )


def ddJ(n, z):
    z = np.asarray(z, dtype=complex)
    escalar = (z.ndim == 0)
    z = np.atleast_1d(z)
    pequeno = np.isclose(z, 0.0)
    z_seguro = np.where(pequeno, 1.0, z)
    out = (
        -(2.0 / z_seguro) * dJ(n, z_seguro)
        + (n * (n + 1) / z_seguro**2 - 1.0) * J(n, z_seguro)
    )
    if np.any(pequeno):
        limite = -1.0 / 3.0 if n == 0 else (2.0 / 15.0 if n == 2 else 0.0)
        out = np.where(pequeno, limite + 0.0j, out)
    return out[0] if escalar else out


def den_a_l(xM, l, cL, cT, cM, rhoS, rhoM):
    xL = xM * (cM / cL)
    eta = cL / cT
    zL = xL
    zT = eta * zL
    zM = xM
    mu = rhoS * cT**2
    lamb = rhoS * cL**2 - 2 * mu

    den_M = rhoM * (cM**2) * (zM**2) * H(l, zM)
    A_l = (lamb * (zL**2) * J(l, zL) - 2 * mu * (zL**2) * ddJ(l, zL)) / den_M
    B_l = -(2 * mu * l * (l + 1) * (zT * dJ(l, zT) - J(l, zT))) / den_M
    den_M_tilde = zM * dH(l, zM)
    A_l_tilde = (zL * dJ(l, zL)) / den_M_tilde
    B_l_tilde = (l * (l + 1) * J(l, zT)) / den_M_tilde
    num_R = 2 * (J(l, zL) - zL * dJ(l, zL))
    den_R = (zT**2) * ddJ(l, zT) + (l * (l + 1) - 2) * J(l, zT)
    R_l = num_R / den_R
    return (A_l - A_l_tilde) - (B_l_tilde - B_l) * R_l


def G(z, l):
    x = np.abs(z)
    return x*J(l, x)*dJ(l, x) + (x**2/2)*(J(l, x)**2 - J(l-1, x)*J(l+1, x))


def coefs(xM, l_max, cL, cT, cM, rhoS, rhoM):
    zL = xM * (cM / cL)
    eta = cL / cT
    zT = eta * zL
    zM = xM
    mu = rhoS * (cT**2)
    lamb = rhoS * (cL**2) - 2 * mu

    a_l = np.zeros(l_max + 1, dtype=complex)
    b_l = np.zeros(l_max + 1, dtype=complex)

    for l in range(l_max + 1):
        den_M = rhoM * (cM**2) * (zM**2) * H(l, zM)
        A_l = (lamb*(zL**2)*J(l,zL) - 2*mu*(zL**2)*ddJ(l,zL)) / den_M
        B_l = -(2*mu*l*(l+1)*(zT*dJ(l,zT)-J(l,zT))) / den_M
        C_l = -J(l,zM)/H(l,zM)

        den_M_tilde = zM*dH(l,zM)
        A_l_tilde = (zL*dJ(l,zL))/den_M_tilde
        B_l_tilde = (l*(l+1)*J(l,zT))/den_M_tilde
        C_l_tilde = -dJ(l,zM)/dH(l,zM)

        num_R = 2*(J(l,zL) - zL*dJ(l,zL))
        den_R = (zT**2)*ddJ(l,zT) + (l*(l+1)-2)*J(l,zT)
        R_l = num_R/den_R if den_R != 0 else 0j

        num_a = C_l_tilde - C_l
        den_a = (A_l - A_l_tilde) - (B_l_tilde - B_l)*R_l
        a_l[l] = num_a/den_a if den_a != 0 else 0j
        b_l[l] = R_l * a_l[l]

    return a_l, b_l


def energia_total_elastica(xM_arr, l_max, cL, cT, cM, rhoS, rhoM):
    xM_arr = np.atleast_1d(xM_arr)
    saida = np.zeros(len(xM_arr))
    for idx, xM in enumerate(xM_arr):
        a_l, b_l = coefs(xM, l_max, cL, cT, cM, rhoS, rhoM)
        zL = (cM / cL) * xM
        xL = np.abs(zL)
        eta = cL / cT
        zT = eta * zL
        mu = rhoS * cT**2
        lamb = rhoS * cL**2 - 2*mu

        dois_T = 0.0
        IS_sum = 0.0
        for l in range(l_max + 1):
            a, b = a_l[l], b_l[l]
            L = l * (l + 1)
            termo_a = xL * G(zL, l)
            termo_b = xL * (J(l, zT)*J(l, zT) + G(zT, l))
            termo_cruzado = xL * J(l, zL) * J(l, zT)
            dois_T += 2*(2*l + 1) * (
                np.abs(a)**2 * termo_a
                + np.abs(b)**2 * L * termo_b
                + 2*L * np.real(a * np.conj(b) * termo_cruzado)
            )
            fator1 = np.conj(a)*dJ(l, zL) + np.conj(b)*(L/zL)*J(l, zT)
            fator2 = a*zL**2*(lamb*J(l,zL) - 2*mu*ddJ(l,zL)) - 2*mu*L*b*(zT*dJ(l,zT) - J(l,zT))
            IS_sum += np.real((2*l + 1) * fator1 * fator2)

        IS_termo = -IS_sum / (rhoS * cL * cL)
        saida[idx] = np.real(dois_T) + IS_termo
    return saida


def contribuicoes_por_l_elastico(xM_arr, l_max, cL, cT, cM, rhoS, rhoM):
    xM_arr = np.atleast_1d(xM_arr)
    n = len(xM_arr)
    contrib = np.zeros((l_max + 1, n))
    for i, xM in enumerate(xM_arr):
        a_l, b_l = coefs(xM, l_max, cL, cT, cM, rhoS, rhoM)
        zL = (cM / cL) * xM
        xL = abs(zL)
        eta = cL / cT
        zT = eta * zL
        mu = rhoS * cT**2
        lamb = rhoS * cL**2 - 2 * mu
        for l in range(l_max + 1):
            a, b = a_l[l], b_l[l]
            L = l * (l + 1)
            termo_a = xL * G(zL, l)
            termo_b = xL * (J(l, zT) * J(l, zT) + G(zT, l))
            termo_cruzado = xL * J(l, zL) * J(l, zT)
            dois_T = 2 * (2 * l + 1) * (
                abs(a)**2 * termo_a + abs(b)**2 * L * termo_b
                + 2 * L * np.real(a * np.conj(b) * termo_cruzado)
            )
            fator1 = np.conj(a) * dJ(l, zL) + np.conj(b) * (L / zL) * J(l, zT)
            fator2 = a * zL**2 * (lamb * J(l, zL) - 2 * mu * ddJ(l, zL)) - 2 * mu * L * b * (zT * dJ(l, zT) - J(l, zT))
            IS_termo = -np.real((2 * l + 1) * fator1 * fator2 / (rhoS * cL * cL))
            contrib[l, i] = np.real(dois_T) + IS_termo
    return contrib


def _malha_e_lmax_efetivo(cL, cT, rhoS, cM=CM_PADRAO, rhoM=RHOM_PADRAO, l_maximo_absoluto=10,
                           xM_min=1e-3, xM_max=1.0, n_scan=3000, razao_minima=1e-3):
    """Faz a varredura de contribuicoes_por_l_elastico UMA VEZ so' e devolve
    tudo que da pra tirar dela: a malha, a energia total (soma exata dos l,
    conferido numericamente == energia_total_elastica), e o l_max efetivo.
    Isso evita rodar duas vezes a parte mais cara (coefs() ponto a ponto)
    so' pra descobrir l_max de um lado e a curva de energia do outro."""
    xM_vals = np.linspace(xM_min, xM_max, n_scan)
    contrib = contribuicoes_por_l_elastico(xM_vals, l_maximo_absoluto, cL, cT, cM, rhoS, rhoM)
    w_total = contrib.sum(axis=0)
    maximo_por_l = np.max(np.abs(contrib), axis=1)
    maximo_global = maximo_por_l.max() if maximo_por_l.max() > 0 else 1.0
    ls_significativos = np.where(maximo_por_l / maximo_global > razao_minima)[0]
    l_max_efetivo = int(ls_significativos.max()) if len(ls_significativos) > 0 else 0
    return xM_vals, w_total, l_max_efetivo


RAZAO_TRIAGEM_MINIMA = 5.0


PROMINENCIA_MALHA_MINIMA = 0.75


def tem_potencial_ressonancia_elastico(xM_vals, w_total, razao_minima=RAZAO_TRIAGEM_MINIMA):
    """Decide, a partir da energia total ja calculada numa malha grosseira,
    se vale a pena rodar a busca cara (raiz complexa por l). Criterio
    frouxo de proposito (mesmo espirito do core-shell): melhor um falso
    positivo (gasta tempo a toa) do que um falso negativo (perde pico)."""
    finito = np.isfinite(w_total) & (np.abs(w_total) > 0.0)
    if not np.any(finito):
        return False, 0.0
    picos_idx, _ = find_peaks(np.abs(w_total), prominence=0.0)
    if len(picos_idx) == 0:
        return False, 0.0
    fundo = max(float(np.median(np.abs(w_total)[finito])), np.finfo(float).tiny)
    razao = float(np.max(np.abs(w_total)[picos_idx]) / fundo)
    return razao > razao_minima, razao


def picos_por_malha_grosseira_elastico(xM_vals, w_total, prominencia_minima=PROMINENCIA_MALHA_MINIMA):
    """Conta/mede picos direto na malha (find_peaks no log10 de |energia|,
    largura a meia altura via peak_widths) -- resultado sempre disponivel,
    sem depender da raiz complexa por l. Mesmo metodo ja usado no detector
    core-shell (ver _picos_por_malha_grosseira la')."""
    w_abs = np.abs(w_total)
    finito = np.isfinite(w_abs) & (w_abs > 0.0)
    if np.count_nonzero(finito) < 5:
        return []
    piso = max(float(np.min(w_abs[finito])) * 1e-14, np.finfo(float).tiny)
    w_seguro = np.where(finito, w_abs, piso)
    log_w = np.log10(np.maximum(w_seguro, piso))

    idx, _ = find_peaks(log_w, prominence=prominencia_minima)
    idx = idx[(idx >= 3) & (idx <= len(xM_vals) - 4)]
    if len(idx) == 0:
        return []

    _, _, lips, rips = peak_widths(w_abs, idx, rel_height=0.5)
    eixo = np.arange(len(xM_vals), dtype=float)
    xl = np.interp(lips, eixo, xM_vals)
    xr = np.interp(rips, eixo, xM_vals)

    picos = []
    for k, i in enumerate(idx):
        largura = float(xr[k] - xl[k])
        altura = float(w_abs[i])
        if not (np.isfinite(largura) and largura > 0.0 and np.isfinite(altura) and altura > 0.0):
            continue
        picos.append({
            "xM_pico": float(xM_vals[i]), "altura": altura, "largura": largura,
            "Q": float(xM_vals[i]) / largura, "origem": "malha_grosseira",
        })
    return picos


def candidatos_reais_para_l(l, cL, cT, cM, rhoS, rhoM, xM_min=1e-3, xM_max=1.0,
                             n_scan=3000, prominencia=0.02):
    xM_vals = np.linspace(xM_min, xM_max, n_scan)
    modulos = np.array([abs(den_a_l(x, l, cL, cT, cM, rhoS, rhoM)) for x in xM_vals])
    log_inv = -np.log10(np.clip(modulos, 1e-300, None))
    minimos_idx, _ = find_peaks(log_inv, prominence=prominencia)
    return xM_vals[minimos_idx]


def _significativo(raiz, l_max_total, cL, cT, cM, rhoS, rhoM,
                    fator_janela=8.0, razao_minima=1.5):
    largura_esperada = max(abs(raiz.imag) * 2, 1e-12)
    janela = fator_janela * largura_esperada
    xM_centro = raiz.real
    xM_todos = np.array([xM_centro - janela, xM_centro, xM_centro + janela])
    try:
        W_todos = energia_total_elastica(xM_todos, l_max_total, cL, cT, cM, rhoS, rhoM)
    except Exception:
        return False
    altura_centro = abs(W_todos[1])
    altura_bordas = max(abs(W_todos[0]), abs(W_todos[2]), 1e-300)
    return (altura_centro / altura_bordas) > razao_minima


def raizes_complexas_para_l(l, l_max_total, cL, cT, cM, rhoS, rhoM, xM_min=1e-3, xM_max=1.0,
                             n_scan=3000, offset_im=-1e-4, tol_aceitar=0.01,
                             prominencia=0.02, tol_convergencia=1e-6):
    candidatos = candidatos_reais_para_l(l, cL, cT, cM, rhoS, rhoM, xM_min, xM_max,
                                          n_scan, prominencia)
    raizes = []
    for xM_c in candidatos:
        try:
            z0 = xM_c + offset_im * 1j
            raiz = newton(lambda z: den_a_l(z, l, cL, cT, cM, rhoS, rhoM),
                           z0, tol=1e-12, maxiter=100)
            residuo = abs(den_a_l(raiz, l, cL, cT, cM, rhoS, rhoM))
            escala = abs(den_a_l(xM_c, l, cL, cT, cM, rhoS, rhoM))
            convergiu = residuo < tol_convergencia * max(escala, 1e-300)
            im_ok = raiz.imag < -1e-10
            perto = abs(raiz.real - xM_c) < tol_aceitar
            if not (convergiu and im_ok and perto):
                continue
            if not _significativo(raiz, l_max_total, cL, cT, cM, rhoS, rhoM):
                continue
            raizes.append(raiz)
        except Exception:
            continue
    return sorted(set(raizes), key=lambda r: r.real)


def _deduplicar(picos):
    """Ordena os picos por xM, sem fundir nada por proximidade.

    A versao anterior (com tolerancia, relativa ou absoluta) sempre corre
    o risco de fundir por engano duas ressonancias genuinamente distintas
    que calham de ficar proximas em xM (por exemplo, l's diferentes cujas
    raizes quase se cruzam) -- qualquer tolerancia > 0 tem essa
    possibilidade, so' muda o quao raro e'. Sem tolerancia nenhuma, cada
    raiz encontrada (e que passou em todos os criterios de aceitacao em
    raizes_complexas_para_l: convergencia, parte imaginaria, proximidade
    do candidato, significancia energetica) conta como um pico proprio."""
    if not picos:
        return picos
    return sorted(picos, key=lambda p: p["xM_pico"])


def contar_picos_elastico(cL, cT, rhoS, cM=CM_PADRAO, rhoM=RHOM_PADRAO, l_max=10,
                           xM_min=1e-3, xM_max=1.0, usar_triagem=True, evitar_overflow=True):
    """Malha de x_M: 0 a 1, 3000 pontos (mantido como sempre).
    Pode retornar lista vazia (N_res=0) -- isso eh um resultado
    valido e tem que ser tratado explicitamente por quem chama.

    Se usar_triagem=True (padrao), antes de rodar a busca de raiz
    complexa (cara: Newton por candidato, por l), reaproveita a MESMA
    varredura que ja calcula l_max_efetivo (nao roda de novo) e decide se
    vale a pena continuar. Se a triagem disser que nao ha ressonancia
    nenhuma, retorna rapido -- sem procurar raiz nenhuma.

    Se evitar_overflow=True (padrao), antes de tentar a raiz complexa,
    risco_overflow_elastico decide (formula fechada em cL, cT) se essa
    combinacao provavelmente estoura float64 durante o Newton (mesmo
    mecanismo do core-shell: as funcoes de Bessel/Hankel crescem
    exponencialmente longe do eixo real). Se sim, a busca de raiz e'
    pulada e o resultado vem direto da malha grosseira (mesma marcada
    origem="malha_grosseira"), que so' usa xM real e nao sofre desse
    problema. Nos dois casos (raiz validada ou fallback pela malha), os
    picos identificados direto na malha ficam sempre anexados em cada
    pico como picos[i]["picos_malha"] (mesmo metodo do core-shell)."""
    xM_vals, w_total, l_max_efetivo = _malha_e_lmax_efetivo(
        cL, cT, rhoS, cM, rhoM, l_maximo_absoluto=l_max,
        xM_min=xM_min, xM_max=xM_max,
    )
    picos_malha = picos_por_malha_grosseira_elastico(xM_vals, w_total)

    if usar_triagem:
        potencialmente_ressonante, razao_triagem = tem_potencial_ressonancia_elastico(xM_vals, w_total)
        if not potencialmente_ressonante:
            return []

    if evitar_overflow:
        risco, amp_L, amp_T = risco_overflow_elastico(cL, cT, cM)
        if risco:
            picos_fallback = [dict(p) for p in picos_malha]
            for p in picos_fallback:
                p["l"] = None
                p["l_max_efetivo"] = l_max_efetivo
            picos_fallback.sort(key=lambda p: p["xM_pico"])
            for i, p in enumerate(picos_fallback):
                p["delta_x"] = (picos_fallback[i + 1]["xM_pico"] - p["xM_pico"]
                                if i < len(picos_fallback) - 1 else float("nan"))
                p["picos_malha"] = picos_malha
            return picos_fallback

    picos = []
    for l in range(l_max_efetivo + 1):
        raizes = raizes_complexas_para_l(l, l_max, cL, cT, cM, rhoS, rhoM,
                                          xM_min=xM_min, xM_max=xM_max)
        for r in raizes:
            altura = float(np.real(energia_total_elastica(
                np.array([r.real]), l_max, cL, cT, cM, rhoS, rhoM)[0]))
            Q = abs(r.real / (2 * r.imag))
            largura = abs(r.real / Q) if Q not in (0, float('inf')) else float('nan')
            picos.append({"xM_pico": r.real, "altura": altura, "largura": largura, "Q": Q,
                           "l": l, "l_max_efetivo": l_max_efetivo, "origem": "polo_validado"})
    picos = _deduplicar(picos)
    picos.sort(key=lambda p: p["xM_pico"])
    for i, p in enumerate(picos):
        p["delta_x"] = picos[i + 1]["xM_pico"] - p["xM_pico"] if i < len(picos) - 1 else float("nan")
        p["picos_malha"] = picos_malha
    return picos


LIMIAR_AMP_OVERFLOW = 40000.0


def risco_overflow_elastico(cL, cT, cM=CM_PADRAO, limiar=LIMIAR_AMP_OVERFLOW):
    """Estimativa fechada (nao roda nada) de se a busca de raiz complexa
    provavelmente estoura float64 (ou pior, cancela tudo silenciosamente)
    para esses cL, cT. amp_L = cM/cL, amp_T = cM/cT -- quanto maior, mais
    uma pequena excursao no plano complexo vira um argumento gigante nas
    funcoes de Bessel/Hankel. Na pratica, dentro do hipercubo (beta<=1000,
    logo amp_L<=1000, bem abaixo do limiar), quem manda e' amp_T, porque
    c_T e' sorteado direto e pode chegar bem perto de 0."""
    amp_L = cM / cL
    amp_T = cM / cT
    risco = (amp_L > limiar) or (amp_T > limiar)
    return bool(risco), float(amp_L), float(amp_T)


# =========================================================
# CARGA DOS CHECKPOINTS E MONTAGEM DOS MAPAS
# =========================================================

def preparar_ambiente():
    """Usa exclusivamente a pasta visco_final definida acima."""
    if not Path(PASTA).is_dir():
        raise FileNotFoundError(
            f"A pasta de dados nao existe: {PASTA}. "
            "No Colab, confirme que Meu Drive/visco_final existe e esta montado."
        )
    os.makedirs(FIG_DIR, exist_ok=True)
    os.makedirs(TAB_DIR, exist_ok=True)
    print("Pasta de dados:", PASTA)
    print("Pasta de resultados:", SAIDA)


def localizar_checkpoints():
    """Busca NPZ/CSV nas subpastas do modelo em visco_final, sem caminhos antigos.

    Prefere o checkpoint do gerador (hipercubo/checkpoint) e arquivos resumo;
    nao procura arquivos dentro de resultados, cache ou pastas antigas do Drive.
    """
    raiz = Path(PASTA)
    nomes = {
        "elastico": ("elastico", "elastic"),
        "voigt": ("voigt", "voight", "kelvin"),
        "maxwell": ("maxwell",),
    }
    escolhidos = {}
    for modelo, termos in nomes.items():
        candidatos = []
        for ext in ("*.npz", "*.csv"):
            for arquivo in raiz.rglob(ext):
                if any(p.lower() in {"resultados", "figuras", "tabelas", "cache_eta_exato"}
                       for p in arquivo.relative_to(raiz).parts[:-1]):
                    continue
                # Deve estar numa subpasta do modelo ou ter nome do modelo.
                caminho_relativo = str(arquivo.relative_to(raiz)).lower()
                if not any(t in caminho_relativo for t in termos):
                    continue
                nome_arq = arquivo.stem.lower()
                nome_arq_norm = nome_arq.replace("voight", "voigt")
                if any(z in nome_arq_norm for z in ("grafico", "figura", "plot", "picos", "cache_")):
                    continue
                pontuacao = (
                    100 if nome_arq_norm == f"hipercubo_{modelo}" else
                    85 if "hipercubo" in nome_arq_norm else
                    75 if "checkpoint" in nome_arq_norm else
                    65 if "resumo" in nome_arq_norm else 20
                )
                if arquivo.suffix.lower() == ".npz":
                    pontuacao += 5
                candidatos.append((pontuacao, arquivo.stat().st_mtime, arquivo))
        candidatos.sort(key=lambda x:(x[0],x[1]),reverse=True)
        escolhidos[modelo] = str(candidatos[0][2]) if candidatos else None
        if candidatos:
            print(f"[{modelo}] arquivo selecionado: {escolhidos[modelo]}")
            if len(candidatos) > 1:
                print(f"[{modelo}] outros candidatos ignorados: {len(candidatos)-1}")
        else:
            print(f"[{modelo}] nenhum NPZ/CSV encontrado em {PASTA} (subpastas do modelo).")
    return escolhidos


def _ler_checkpoint_npz(data):
    chaves = set(data.files)
    if 'picos_por_curva' in chaves:
        brutos = data['picos_por_curva'].item()
        linhas = []
        for registro in brutos.values():
            if not isinstance(registro, dict):
                continue
            a = float(registro.get('alpha', np.nan))
            b = float(registro.get('beta', np.nan))
            eta = float(registro.get('eta', np.nan))
            n = registro.get('N_res', None)
            if n is None:
                n = len(registro.get('picos', []))
            linhas.append((a, b, eta, float(n)))
        df = pd.DataFrame(linhas, columns=['alpha', 'beta', 'eta', 'N_res'])
    else:
        if '_indice' in chaves:
            indices = np.asarray(data['_indice'])
            _, pos = np.unique(indices, return_index=True)
            pos = np.sort(pos)
        else:
            pos = np.arange(len(data[next(iter(chaves))]))
        df = pd.DataFrame({chave: np.asarray(data[chave])[pos]
                           for chave in chaves if np.asarray(data[chave]).ndim == 1
                           and len(data[chave]) > int(pos.max())})
    return df


def carregar_resumo_npz(caminho, nome):
    """Le NPZ ou CSV do gerador, deduplicando linhas de picos por curva."""
    if str(caminho).lower().endswith('.csv'):
        df = pd.read_csv(caminho)
        if '_indice' in df.columns:
            df = df.drop_duplicates('_indice').copy()
    else:
        with np.load(caminho, allow_pickle=True) as data:
            df = _ler_checkpoint_npz(data)
    if df.empty:
        raise ValueError(f'Checkpoint vazio: {caminho}')
    if 'status' in df.columns:
        df = df.loc[df['status'].astype(str) == 'ok'].copy()
    if 'eta' not in df:
        if {'cL0','cT0'}.issubset(df):
            df['eta'] = df['cL0'].astype(float)/df['cT0'].astype(float)
        elif {'cL','cT'}.issubset(df):
            df['eta'] = df['cL'].astype(float)/df['cT'].astype(float)
        else:
            raise KeyError(f'{nome}: falta eta (ou cL/cT).')
    if 'N_res' not in df:
        if 'N' in df:
            df['N_res'] = df['N']
        else:
            raise KeyError(f'{nome}: falta N_res.')
    if {'alpha','beta'}.issubset(df.columns):
        # No gerador: alpha = rhoM/rhoS, beta = cM/cL.
        df['c_razao'] = 1.0 / pd.to_numeric(df['beta'])
        df['rho_ratio'] = 1.0 / pd.to_numeric(df['alpha'])
    elif {'cL','cM','rhoS','rhoM'}.issubset(df.columns):
        df['c_razao'] = df['cL']/df['cM']
        df['rho_ratio'] = df['rhoS']/df['rhoM']
    elif not {'c_razao','rho_ratio'}.issubset(df.columns):
        raise KeyError(f'{nome}: falta (alpha,beta), (cL,cM,rhoS,rhoM) ou (c_razao,rho_ratio).')
    for nome_col in ('c_razao','rho_ratio','eta','N_res'):
        df[nome_col] = pd.to_numeric(df[nome_col], errors='coerce')
    m = (np.isfinite(df['c_razao']) & (df['c_razao'] > 0) &
         np.isfinite(df['rho_ratio']) & (df['rho_ratio'] > 0) &
         np.isfinite(df['eta']) & (df['eta'] >= 2/np.sqrt(3)-1e-7) &
         np.isfinite(df['N_res']) & (df['N_res'] >= 0))
    df = df.loc[m, ['c_razao','rho_ratio','eta','N_res']].rename(columns={'N_res':'N'}).copy()
    print(f'[{nome}] {len(df)} curvas validas: {caminho}')
    return df


class Grade:
    def __init__(self, log_x, log_y):
        self.LOG_X, self.LOG_Y = log_x, log_y
        self.NX = int(round((log_x[1]-log_x[0])/LARGURA_BIN))
        self.NY = int(round((log_y[1]-log_y[0])/LARGURA_BIN))

    def celula_x(self, valor):
        return (np.log10(valor)-self.LOG_X[0])/LARGURA_BIN-0.5

    def celula_y(self, valor):
        return (np.log10(valor)-self.LOG_Y[0])/LARGURA_BIN-0.5


def mapa_binado(df, gr, preencher=False):
    ex = np.linspace(*gr.LOG_X, gr.NX+1)
    ey = np.linspace(*gr.LOG_Y, gr.NY+1)
    cont = np.zeros((gr.NX, gr.NY), dtype=int)
    mapa = np.full((gr.NX, gr.NY), -np.inf)
    if df.empty:
        return mapa, cont
    lx = np.log10(df['c_razao'].to_numpy(dtype=float))
    ly = np.log10(df['rho_ratio'].to_numpy(dtype=float))
    n = df['N'].to_numpy(dtype=float)
    dentro = (np.isfinite(lx) & np.isfinite(ly) & np.isfinite(n) & (n>=0)
              & (lx>=ex[0]) & (lx<=ex[-1]) & (ly>=ey[0]) & (ly<=ey[-1]))
    lx, ly, n = lx[dentro], ly[dentro], n[dentro]
    if not len(n):
        mapa[:] = np.nan
        return mapa, cont
    ix = np.clip(np.searchsorted(ex,lx,side='right')-1,0,gr.NX-1)
    iy = np.clip(np.searchsorted(ey,ly,side='right')-1,0,gr.NY-1)
    np.add.at(cont,(ix,iy),1)
    np.maximum.at(mapa, (ix,iy), n)
    mapa[cont==0] = np.nan
    if preencher and np.any(cont==0):
        cx, cy = np.meshgrid(0.5*(ex[:-1]+ex[1:]), 0.5*(ey[:-1]+ey[1:]), indexing='ij')
        faltantes = cont==0
        _, ids = cKDTree(np.column_stack((lx,ly))).query(np.column_stack((cx[faltantes],cy[faltantes])))
        mapa[faltantes] = n[ids]
    return mapa, cont


def segmentos_fronteira(mask, conhecidos=None):
    if conhecidos is None:
        conhecidos = np.ones_like(mask, dtype=bool)
    seg=[]
    nx,ny=mask.shape
    for i in range(nx):
        for j in range(ny):
            if not mask[i,j]:
                continue
            xl,xr,yb,yt=i-.5,i+.5,j-.5,j+.5
            if i==0 or (conhecidos[i-1,j] and not mask[i-1,j]): seg.append(((xl,yb),(xl,yt)))
            if i==nx-1 or (conhecidos[i+1,j] and not mask[i+1,j]): seg.append(((xr,yb),(xr,yt)))
            if j==0 or (conhecidos[i,j-1] and not mask[i,j-1]): seg.append(((xl,yb),(xr,yb)))
            if j==ny-1 or (conhecidos[i,j+1] and not mask[i,j+1]): seg.append(((xl,yt),(xr,yt)))
    return seg


def caminho_cache_eta(eta):
    pasta=Path(FIG_DIR)/'cache_eta_exato'
    pasta.mkdir(parents=True,exist_ok=True)
    slug=f'{eta:g}'.replace('.','p')
    return pasta/f'elastico_eta_{slug}_bin{str(LARGURA_BIN).replace(".","p")}.csv'


def calcular_eta_elastico(eta, gr):
    """Calcula em eta exato, usando o detector do elastico(1).py incorporado acima."""
    destino = caminho_cache_eta(eta)
    colunas = ['ix','iy','eta','c_razao','rho_ratio','N','status']
    if destino.is_file():
        cache=pd.read_csv(destino)
        if not set(colunas).issubset(cache.columns):
            raise ValueError(f'Cache incompatível: {destino}')
    else:
        cache=pd.DataFrame(columns=colunas)
    # Resultados com falha sao reprocessados em execucoes futuras.
    ok=cache.loc[cache['status'].eq('ok')]
    feitos=set(zip(ok['ix'].astype(int),ok['iy'].astype(int)))
    faltam=[(i,j) for i in range(gr.NX) for j in range(gr.NY) if (i,j) not in feitos]
    if faltam:
        print(f'[Elastico eta={eta:g}] calculando {len(faltam)}/{gr.NX*gr.NY} celulas')
    novos=[]
    for k,(i,j) in enumerate(faltam,1):
        cx = 10**(gr.LOG_X[0]+(i+.5)*LARGURA_BIN)
        ry = 10**(gr.LOG_Y[0]+(j+.5)*LARGURA_BIN)
        cL = CM_PADRAO*cx
        cT = cL/eta
        rhoS= RHOM_PADRAO*ry
        try:
            if TEMPO_LIMITE_CURVA and hasattr(signal,'SIGALRM'):
                with curve_timeout(TEMPO_LIMITE_CURVA):
                    picos=contar_picos_elastico(cL,cT,rhoS, cM=CM_PADRAO,
                                                rhoM=RHOM_PADRAO,l_max=L_MAX,
                                                xM_min=1e-3,xM_max=1.0)
            else:
                picos=contar_picos_elastico(cL,cT,rhoS,cM=CM_PADRAO,
                                            rhoM=RHOM_PADRAO,l_max=L_MAX,
                                            xM_min=1e-3,xM_max=1.0)
            N,status=len(picos),'ok'
        except Exception as exc:
            N,status=np.nan,f'{type(exc).__name__}: {str(exc)[:80]}'
        novos.append(dict(ix=i,iy=j,eta=eta,c_razao=cx,rho_ratio=ry,N=N,status=status))
        if k % SALVAR_CADA==0 or k==len(faltam):
            # Substitui os registros de falhas antigas das mesmas celulas.
            novas=pd.DataFrame(novos)
            indices=set(zip(novas['ix'].astype(int),novas['iy'].astype(int)))
            cache=cache.loc[[ (int(a),int(b)) not in indices for a,b in zip(cache['ix'],cache['iy']) ]]
            cache=novas.copy() if cache.empty else pd.concat([cache,novas],ignore_index=True)
            cache.to_csv(destino,index=False)
            novos=[]
            print(f'  eta={eta:g}: {k}/{len(faltam)} novas celulas; {sum(cache.status=="ok")} validas')
    validos=cache.loc[(cache['status']=='ok') & pd.to_numeric(cache['N'],errors='coerce').notna()].copy()
    validos['N']=pd.to_numeric(validos['N'])
    if validos.empty:
        raise RuntimeError(f'Nenhuma celula convergiu em eta={eta:g}; ver cache {destino}')
    falhas=(cache['status']!='ok').sum()
    if falhas:
        print(f'[AVISO] eta={eta:g}: {falhas} falhas/tempos excedidos. Areas ficam cinza, nao N=0')
    validos.attrs['calculado_eta_exato']=True
    return validos


def obter_eta_elastico(df,eta):
    grade=Grade(*LIMITES_MAPA['elastico'])
    if df is not None and not df.empty:
        exato=df.loc[np.isclose(df['eta'],eta,rtol=0,atol=ETA_EXATO_TOL)].copy()
        if len(exato):
            _,cont=mapa_binado(exato,grade,preencher=False)
            cobertura=np.mean(cont>0)
            if cobertura>=COBERTURA_MINIMA:
                print(f'[Elastico eta={eta:g}] banco exato, cobertura {cobertura:.1%}')
                exato.attrs['calculado_eta_exato']=False
                return exato
    print(f'[Elastico eta={eta:g}] sem cobertura exata: recalculando pelo modelo incorporado')
    return calcular_eta_elastico(eta,grade)


def fatia_eta(df,k):
    lo,hi=ETA_FAIXAS[k]
    return df.loc[(df['eta']>=lo*(1-1e-9)) & (df['eta']<hi)].copy()


def texto_decimal(t, pt):
    if t<1:
        v=f'{t:.4f}'.rstrip('0').rstrip('.')
    else:
        v=f'{t:g}'
    return v.replace('.',',') if pt else v


def desenhar_mapa(ax, mapa, gr, idioma, escala_max):
    pt=idioma=='pt'
    norm=Normalize(vmin=0.0,vmax=np.log10(1.0+max(escala_max,1)))
    for i in range(gr.NX):
        for j in range(gr.NY):
            v=mapa[i,j]
            cor=RESONANCE_CMAP(norm(np.log10(1+v))) if np.isfinite(v) else '#D9D9D9'
            ax.add_patch(Rectangle((i-.5,j-.5),1,1,facecolor=cor,
                                   edgecolor=CELL_LINECOLOR,linewidth=CELL_LINEWIDTH,zorder=1))
    tx=10.0**np.arange(np.ceil(gr.LOG_X[0]),gr.LOG_X[1]+1e-9)
    ty=10.0**np.arange(np.ceil(gr.LOG_Y[0]),gr.LOG_Y[1]+1e-9)
    ax.set_xticks(gr.celula_x(tx)); ax.set_xticklabels([texto_decimal(t,pt) for t in tx])
    ax.set_yticks(gr.celula_y(ty)); ax.set_yticklabels([texto_decimal(t,pt) for t in ty])
    ax.set_xlabel(r'$c_L/c_M$',fontsize=23,labelpad=9)
    ax.set_ylabel(r'$\rho_S/\rho_M$',fontsize=23,labelpad=10)
    ax.set_xlim(-.5,gr.NX-.5); ax.set_ylim(-.5,gr.NY-.5)
    ax.set_aspect('equal',adjustable='box')
    return norm


def barra_cores(fig, cax, norm, escala_max, idioma):
    pt=idioma=='pt'
    nmax=max(int(np.ceil(escala_max)),1)
    valores=np.unique(np.array([0]+[10**k for k in range(1,6) if 10**k<.8*nmax]+[nmax],dtype=int))
    sm=cm.ScalarMappable(norm=norm,cmap=RESONANCE_CMAP)
    sm.set_array([])
    cb=fig.colorbar(sm,cax=cax,ticks=np.log10(1+valores))
    cb.ax.set_yticklabels([str(int(v)) for v in valores])
    cb.set_label('Ressonâncias' if pt else 'Resonances',fontsize=20,labelpad=18,rotation=270)
    return cb


def estilizar_eixos(ax,cb):
    for eixo in (ax,cb.ax):
        eixo.tick_params(axis='both',labelsize=15,width=1.25,length=4.5,direction='out')
        for borda in eixo.spines.values():
            borda.set_visible(True); borda.set_edgecolor(BOX_COLOR); borda.set_linewidth(1.8)


def criar_figura(gr):
    altura=.70; largura=altura*gr.NX/gr.NY
    fig=plt.figure(figsize=(10.5,10.5),facecolor='white')
    ax=fig.add_axes([.12,.13,largura,altura])
    cax=fig.add_axes([.12+largura+.02,.13,.03,altura])
    return fig,ax,cax


def salvar(fig,nome,mostrar):
    os.makedirs(FIG_DIR,exist_ok=True)
    for ext in ('png','pdf','eps'):
        fig.savefig(str(Path(FIG_DIR)/(nome+'.'+ext)),dpi=600,bbox_inches='tight')
    if mostrar:
        plt.show()
    plt.close(fig)


def mapa_elastico(dados,idioma,escala_max,mostrar):
    gr=Grade(*LIMITES_MAPA['elastico'])
    mats={}
    for eta,df in dados.items():
        mats[eta],_=mapa_binado(df,gr,preencher=(
            PREENCHER_VAZIOS_BANCO and not df.attrs.get('calculado_eta_exato',False)))
    with plt.rc_context({'font.family':'sans-serif'}):
        fig,ax,cax=criar_figura(gr)
        norm=desenhar_mapa(ax,mats[1.16],gr,idioma,escala_max)
        ax.axvline(gr.celula_x(1.0),**GUIAS)
        ax.axhline(gr.celula_y(1.0),**GUIAS)
        for eta in (1.16,2.0,5.0,10.0):
            mat=mats[eta]
            estilo=ESTILO_FRONTEIRA_BASE if eta==1.16 else FRONTEIRAS_ESTILOS[eta]
            for (a,b) in segmentos_fronteira(mat>0,np.isfinite(mat)):
                ax.plot([a[0],b[0]],[a[1],b[1]],solid_capstyle='butt',
                        zorder=15 if eta==1.16 else 16,**estilo)
        handles=[Line2D([0],[0],label=(('Fronteira de referência\n' if idioma=='pt' else 'Reference frontier\n')+rf'$\eta={eta:g}$'),
                        **FRONTEIRAS_ESTILOS[eta]) for eta in (2.0,5.0,10.0)]
        lg=ax.legend(handles=handles,loc='upper right',bbox_to_anchor=(.985,.83),
                     fontsize=16,frameon=True,framealpha=1,facecolor='white',edgecolor='none',
                     fancybox=False,borderpad=.22,handlelength=2.3,handletextpad=.8,labelspacing=.55)
        lg.set_zorder(100)
        for x,t,align in [(.40,r'$\eta=1{,}16$' if idioma=='pt' else r'$\eta=1.16$','left'),
                          (.955,r'$N_{\mathrm{res}}$','right')]:
            ax.text(x,.965,t,transform=ax.transAxes,fontsize=22,color='white',va='top',ha=align,
                    bbox=dict(facecolor=BOX_COLOR,edgecolor='none',boxstyle='square,pad=0.14'),zorder=111)
        cb=barra_cores(fig,cax,norm,escala_max,idioma)
        estilizar_eixos(ax,cb)
        salvar(fig,f'mapa_fase_elastico_{idioma}',mostrar)


def mapa_separado(df,modelo,faixa,idioma,escala_max,mostrar):
    gr=Grade(*LIMITES_MAPA[modelo])
    mat,_=mapa_binado(df,gr,preencher=PREENCHER_VAZIOS_BANCO)
    with plt.rc_context({'font.family':'sans-serif'}):
        fig,ax,cax=criar_figura(gr)
        norm=desenhar_mapa(ax,mat,gr,idioma,escala_max)
        rotulo=ROT_FAIXA[faixa]+r'\quad $N_{\mathrm{res}}$'
        ax.text(.955,.965,rotulo,transform=ax.transAxes,fontsize=21,color='white',va='top',ha='right',
                bbox=dict(facecolor=BOX_COLOR,edgecolor='none',boxstyle='square,pad=0.14'),zorder=111)
        cb=barra_cores(fig,cax,norm,escala_max,idioma)
        estilizar_eixos(ax,cb)
        salvar(fig,f'mapa_fase_{modelo}_eta{faixa+1}_{idioma}',mostrar)


def executar():
    preparar_ambiente()
    arquivos=localizar_checkpoints()
    bancos={}
    for nome,arquivo in arquivos.items():
        if arquivo is None:
            bancos[nome]=None
            print(f'[AVISO] Checkpoint {nome} ausente; modelo elastico pode ser recalculado, '
                  'mas Voigt/Maxwell exigem seus NPZ.')
        else:
            bancos[nome]=carregar_resumo_npz(arquivo,nome)
    elasticos={e:obter_eta_elastico(bancos['elastico'],e) for e in (1.16,2.0,5.0,10.0)}
    series={}
    for modelo in ('voigt','maxwell'):
        if bancos[modelo] is None:
            continue
        for k in range(3):
            pedaco=fatia_eta(bancos[modelo],k)
            print(f'{modelo}, faixa {k+1}: {len(pedaco)} curvas')
            if len(pedaco):
                series[(modelo,k)]=pedaco
    colecoes=[np.asarray(df['N'],dtype=float) for df in elasticos.values()]
    colecoes += [np.asarray(df['N'],dtype=float) for df in series.values()]
    nmax=max((float(np.nanmax(x)) for x in colecoes if x.size and np.isfinite(x).any()),default=1.0)
    print('Escala de cores compartilhada, N_res maximo:',nmax)
    idiomas=['pt','en'] if GERAR_INGLES else ['pt']
    for idioma in idiomas:
        mapa_elastico(elasticos,idioma,nmax,mostrar=(MOSTRAR_FIGURAS and idioma=='pt'))
        for (modelo,k),df in series.items():
            mapa_separado(df,modelo,k,idioma,nmax,mostrar=(MOSTRAR_FIGURAS and idioma=='pt'))
    print('Figuras em:',FIG_DIR)
    print('Tabelas em:',TAB_DIR)


if __name__ == '__main__':
    executar()
